## CAP6938 AI Knowledge TESTER

### Part 1: 

TEST

In [1]:
import sys
# initialization code
import numpy as np
from keras.datasets import mnist
%matplotlib inline
from matplotlib import pyplot as plt
from scipy import stats
import torch
import torch.nn as nn

print("Python:", sys.version)
print("Executable:", sys.executable)

packages = [
    "torch",
    "torchvision",
    "transformers",
    "datasets",
    "sentence_transformers",
    "PIL",
    "numpy",
    "matplotlib",
]

for package in packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed (version unavailable)")
        print(f"{package:22s} OK   {version}")
    except ImportError as e:
        print(f"{package:22s} MISSING   {e}")

Python: 3.10.4 (tags/v3.10.4:9d38120, Mar 23 2022, 23:13:41) [MSC v.1929 64 bit (AMD64)]
Executable: C:\Python310\python.exe
torch                  OK   2.5.1+cu121
torchvision            OK   0.20.1+cu121
transformers           OK   5.17.0
datasets               OK   4.8.4
sentence_transformers  OK   6.0.1
PIL                    OK   12.2.0
numpy                  OK   1.26.4
matplotlib             OK   3.7.2


In [2]:
# Sets device to "cuda" if a GPU is available  (in Colabs, enable GPU by Edit->Notebook Settings-->Hardware Accelerator=GPU)
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(device) # make sure you're using GPU instance
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version used by torch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

cuda
Torch version: 2.5.1+cu121
CUDA available: True
CUDA version used by torch: 12.1
GPU: NVIDIA GeForce RTX 2070


#### 1a


In [3]:
try:
    import faiss
    print("faiss                  OK  ", faiss.__version__)
except ImportError as e:
    print("faiss                  MISSING  ", e)

faiss                  OK   1.15.1


In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import json
from pathlib import Path
from pprint import pprint

DATA_DIR = Path("../Yelp_JSON")
BUSINESS_FILE = DATA_DIR / "yelp_academic_dataset_business.json"

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    first_business = json.loads(f.readline())

pprint(first_business)

{'address': '1616 Chapala St, Ste 2',
 'attributes': {'ByAppointmentOnly': 'True'},
 'business_id': 'Pns2l4eNsfO8kk83dixA6A',
 'categories': 'Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, '
               'Acupuncture, Health & Medical, Nutritionists',
 'city': 'Santa Barbara',
 'hours': None,
 'is_open': 0,
 'latitude': 34.4266787,
 'longitude': -119.7111968,
 'name': 'Abby Rappoport, LAC, CMQ',
 'postal_code': '93101',
 'review_count': 7,
 'stars': 5.0,
 'state': 'CA'}


In [6]:
import json

restaurant_count = 0
total_businesses = 0
restaurant_cities = {}

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)
        total_businesses += 1

        categories = business.get("categories") or ""
        category_list = [
            category.strip().lower()
            for category in categories.split(",")
        ]

        if "restaurants" in category_list:
            restaurant_count += 1

            city = business.get("city", "Unknown")
            restaurant_cities[city] = restaurant_cities.get(city, 0) + 1

print(f"Total businesses: {total_businesses:,}")
print(f"Restaurants: {restaurant_count:,}")

print("\nTop 10 cities by restaurant count:")
for city, count in sorted(
    restaurant_cities.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]:
    print(f"{city}: {count:,}")

Total businesses: 150,346
Restaurants: 52,268

Top 10 cities by restaurant count:
Philadelphia: 5,852
Tampa: 2,960
Indianapolis: 2,862
Nashville: 2,502
Tucson: 2,466
New Orleans: 2,259
Edmonton: 2,166
Saint Louis: 1,790
Reno: 1,286
Boise: 847


In [7]:
import json
from pprint import pprint

REVIEW_FILE = DATA_DIR / "yelp_academic_dataset_review.json"

# Step 1: Select a Tampa restaurant with at least 100 reviews.
selected_restaurant = None

with BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)

        categories = [
            c.strip().lower()
            for c in (business.get("categories") or "").split(",")
        ]

        if (
            business.get("city") == "Tampa"
            and "restaurants" in categories
            and business.get("review_count", 0) >= 100
        ):
            selected_restaurant = business
            break

if selected_restaurant is None:
    raise ValueError("No matching Tampa restaurant found.")

print("SELECTED RESTAURANT")
print(f"Name: {selected_restaurant['name']}")
print(f"Business ID: {selected_restaurant['business_id']}")
print(f"Categories: {selected_restaurant['categories']}")
print(f"Rating: {selected_restaurant['stars']}")
print(f"Review count: {selected_restaurant['review_count']}")

# Step 2: Retrieve up to five reviews for this restaurant.
reviews = []
business_id = selected_restaurant["business_id"]

with REVIEW_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)

        if review["business_id"] == business_id:
            reviews.append(review)

            if len(reviews) == 5:
                break

print(f"\nFOUND {len(reviews)} REVIEWS\n")

for i, review in enumerate(reviews, start=1):
    print(f"{'=' * 60}")
    print(f"REVIEW {i}")
    print(f"Stars: {review['stars']}")
    print(f"Date: {review['date']}")
    print(f"Text:\n{review['text']}\n")

SELECTED RESTAURANT
Name: O'Briens Irish Pub
Business ID: Ucl9Vo5lwrUmYbV8Dv8X5g
Categories: Bars, Pubs, Nightlife, Arts & Entertainment, Irish, Restaurants, Music Venues
Rating: 4.0
Review count: 108

FOUND 5 REVIEWS

REVIEW 1
Stars: 3.0
Date: 2013-12-08 23:11:18
Text:
I must say that this location is much better than the old location they used to occupy further south in Carrollwood.  There's always plenty of parking and it's a pretty convenient location.  My usual adventures here have been for their annual St. Patrick's Day blowout, which is under a huge tent outdoors, and to watch Gator football games as they are the north Tampa watch location for the Tampa Gator Club.  I've always been met with attentive servers and bartenders every time I have visited.  The food is your average run of the mill Irish bar food, but I think the main attraction is the atmosphere this place provides.

REVIEW 2
Stars: 1.0
Date: 2014-06-30 13:51:09
Text:
I was a huge fan of the old O'Briens with the dive

## Part 2: Tampa restaurant business subset

In [8]:
import json

TAMPA_BUSINESS_FILE = DATA_DIR / "tampa_restaurants.json"

restaurant_count = 0

with BUSINESS_FILE.open("r", encoding="utf-8") as source:
    with TAMPA_BUSINESS_FILE.open("w", encoding="utf-8") as destination:

        for line in source:
            business = json.loads(line)

            categories = [
                category.strip().lower()
                for category in (business.get("categories") or "").split(",")
            ]

            if (
                business.get("city", "").strip().lower() == "tampa"
                and "restaurants" in categories
            ):
                destination.write(json.dumps(business) + "\n")
                restaurant_count += 1

print(f"Tampa restaurants saved: {restaurant_count:,}")
print(f"Output file: {TAMPA_BUSINESS_FILE}")
print(f"File size: {TAMPA_BUSINESS_FILE.stat().st_size / (1024**2):.2f} MB")

Tampa restaurants saved: 2,967
Output file: ..\Yelp_JSON\tampa_restaurants.json
File size: 3.47 MB


#### 2a


In [9]:
TAMPA_REVIEW_FILE = DATA_DIR / "tampa_restaurant_reviews.jsonl"

# Load the IDs of our Tampa restaurants into a set.
tampa_business_ids = set()

with TAMPA_BUSINESS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        business = json.loads(line)
        tampa_business_ids.add(business["business_id"])

print(f"Tampa business IDs loaded: {len(tampa_business_ids):,}")

# Stream through the original Yelp review dataset.
total_reviews = 0
tampa_reviews = 0

with REVIEW_FILE.open("r", encoding="utf-8") as source:
    with TAMPA_REVIEW_FILE.open("w", encoding="utf-8") as destination:

        for line in source:
            review = json.loads(line)
            total_reviews += 1

            if review["business_id"] in tampa_business_ids:
                destination.write(line)
                tampa_reviews += 1

print(f"\nTotal Yelp reviews scanned: {total_reviews:,}")
print(f"Tampa restaurant reviews saved: {tampa_reviews:,}")
print(f"Output file: {TAMPA_REVIEW_FILE}")
print(f"File size: {TAMPA_REVIEW_FILE.stat().st_size / (1024**2):.2f} MB")

Tampa business IDs loaded: 2,967

Total Yelp reviews scanned: 6,990,280
Tampa restaurant reviews saved: 303,691
Output file: ..\Yelp_JSON\tampa_restaurant_reviews.jsonl
File size: 214.43 MB


#### 2b



In [10]:
# Verify that every Tampa review references a valid Tampa restaurant.

total_reviews = 0
valid_reviews = 0
invalid_reviews = 0
restaurants_with_reviews = set()

with TAMPA_REVIEW_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)
        total_reviews += 1

        business_id = review["business_id"]

        if business_id in tampa_business_ids:
            valid_reviews += 1
            restaurants_with_reviews.add(business_id)
        else:
            invalid_reviews += 1

restaurants_without_reviews = tampa_business_ids - restaurants_with_reviews

print("TAMPA DATA INTEGRITY CHECK")
print("-" * 40)
print(f"Total reviews checked:        {total_reviews:,}")
print(f"Valid reviews:                {valid_reviews:,}")
print(f"Invalid reviews:              {invalid_reviews:,}")
print(f"Restaurants with reviews:     {len(restaurants_with_reviews):,}")
print(f"Restaurants without reviews:  {len(restaurants_without_reviews):,}")

if invalid_reviews == 0:
    print("\nPASS: All reviews reference valid Tampa restaurants.")
else:
    print("\nFAIL: Some reviews reference unknown businesses.")

TAMPA DATA INTEGRITY CHECK
----------------------------------------
Total reviews checked:        303,691
Valid reviews:                303,691
Invalid reviews:              0
Restaurants with reviews:     2,967
Restaurants without reviews:  0

PASS: All reviews reference valid Tampa restaurants.


## Part 3: Test utilities

In [11]:
from data_loader import load_restaurants, get_restaurant

restaurants = load_restaurants()

print(f"Restaurants loaded: {len(restaurants):,}")

# Retrieve our previously examined restaurant.
business_id = "Ucl9Vo5lwrUmYbV8Dv8X5g"

restaurant = get_restaurant(restaurants, business_id)

if restaurant:
    print(f"Name: {restaurant['name']}")
    print(f"City: {restaurant['city']}")
    print(f"Rating: {restaurant['stars']}")
    print(f"Reviews: {restaurant['review_count']}")
else:
    print("Restaurant not found.")

Restaurants loaded: 2,967
Name: O'Briens Irish Pub
City: Tampa
Rating: 4.0
Reviews: 108


In [12]:
from review_retrieval import load_reviews, get_reviews

reviews_by_business = load_reviews()

print(f"Restaurants with reviews: {len(reviews_by_business):,}")
print(f"Total reviews loaded: {sum(len(r) for r in reviews_by_business.values()):,}")

# Use the same restaurant from our previous test.
business_id = "Ucl9Vo5lwrUmYbV8Dv8X5g"

restaurant_reviews = get_reviews(reviews_by_business, business_id)

print(f"\nReviews for {restaurants[business_id]['name']}: {len(restaurant_reviews)}")

# Display the first review.
if restaurant_reviews:
    review = restaurant_reviews[0]

    print(f"\nStars: {review['stars']}")
    print(f"Date: {review['date']}")
    print(f"Text:\n{review['text']}")

Restaurants with reviews: 2,967
Total reviews loaded: 303,691

Reviews for O'Briens Irish Pub: 115

Stars: 3.0
Date: 2013-12-08 23:11:18
Text:
I must say that this location is much better than the old location they used to occupy further south in Carrollwood.  There's always plenty of parking and it's a pretty convenient location.  My usual adventures here have been for their annual St. Patrick's Day blowout, which is under a huge tent outdoors, and to watch Gator football games as they are the north Tampa watch location for the Tampa Gator Club.  I've always been met with attentive servers and bartenders every time I have visited.  The food is your average run of the mill Irish bar food, but I think the main attraction is the atmosphere this place provides.


In [13]:
from restaurant_search import search_restaurants

results = search_restaurants(
    restaurants,
    cuisine="Italian",
    min_stars=4.0,
    min_reviews=50
)

print(f"Matching restaurants: {len(results):,}\n")

for business in results[:10]:
    print(
        f"{business['name']:<40} "
        f"Stars: {business['stars']:.1f}  "
        f"Reviews: {business['review_count']}"
    )

Matching restaurants: 73

Mirro's Pizzeria                         Stars: 5.0  Reviews: 111
OLIVIA                                   Stars: 4.5  Reviews: 545
Pane Rustica                             Stars: 4.5  Reviews: 468
Osteria Natalina                         Stars: 4.5  Reviews: 400
Due Amici                                Stars: 4.5  Reviews: 350
Oggi Italian                             Stars: 4.5  Reviews: 345
Piccola Italia Bistro                    Stars: 4.5  Reviews: 289
Vino E Pasta                             Stars: 4.5  Reviews: 271
La Terrazza Ristorante Italiano          Stars: 4.5  Reviews: 210
Magdalena's Pizzeria                     Stars: 4.5  Reviews: 178


In [14]:
# Examine reviews for the top three Italian restaurants.

for business in results[:3]:
    business_id = business["business_id"]
    reviews = get_reviews(reviews_by_business, business_id)

    print("=" * 70)
    print(f"Restaurant: {business['name']}")
    print(f"Rating: {business['stars']}")
    print(f"Available reviews: {len(reviews)}")

    # Display the first two reviews.
    for review in reviews[:2]:
        print(f"\nReview rating: {review['stars']}")
        print(f"Date: {review['date']}")
        print(f"Text: {review['text']}")

    print()

Restaurant: Mirro's Pizzeria
Rating: 5.0
Available reviews: 119

Review rating: 5.0
Date: 2019-01-15 17:24:29
Text: Best NY style pizza in South Tampa! I moved to Tampa just over a year ago and have been searching for some great NY pizza. Mirro's has it! Delicious pizza, friendly staff and wine too!

Review rating: 5.0
Date: 2019-03-02 14:36:24
Text: This has to be the best pizza outside of Brooklyn. I am so glad I found you. It is definitely like going home again. Delicious.

Restaurant: OLIVIA
Rating: 4.5
Available reviews: 561

Review rating: 4.0
Date: 2019-11-30 23:41:09
Text: The menu isn't yet full since they just opened. The waitstaff was very good and the kitchen overall did well, although a little slow. The pastas are made in house and has excellent savory flavor combinations. Our biggest complaint is the $8 truffle fries were mediocre fries that had no flavor of truffle whatsoever... just Parmesan and parsley sprinkles over less than average fries. 

We will definitely be goi

In [15]:
#manually reload library
#import importlib
#import review_retrieval

#importlib.reload(review_retrieval)

#from review_retrieval import search_reviews

In [16]:
from review_retrieval import search_reviews

# Search for reviews relevant to a romantic Italian dinner.
keywords = ["quiet", "romantic", "atmosphere", "service"]

# Examine Pane Rustica, one of our previous search results.
business = next(
    b for b in results if b["name"] == "Pane Rustica"
)

matching_reviews = search_reviews(
    reviews_by_business,
    business["business_id"],
    keywords,
    limit=5
)

print(f"Restaurant: {business['name']}")
print(f"Keywords: {keywords}")
print(f"Matching reviews returned: {len(matching_reviews)}")

for i, review in enumerate(matching_reviews, start=1):
    print("\n" + "=" * 70)
    print(f"Review {i}")
    print(f"Stars: {review['stars']}")
    print(f"Date: {review['date']}")
    print(f"Text:\n{review['text']}")

Restaurant: Pane Rustica
Keywords: ['quiet', 'romantic', 'atmosphere', 'service']
Matching reviews returned: 5

Review 1
Stars: 4.0
Date: 2016-12-21 01:08:10
Text:
My boyfriend took me here for a special date night and I had never been. It was so perfect. The atmosphere is romantic and trendy. The service is fantastic. The food! The food is so so good. They are only open for dinner 4 nights a week so a special menu is created for each night (with the normal menu on the back as well). I had the gnocchi in bolognese and it was to die for. So rich and flavorful. I absolutely loved my evening here. Definitely will be back!

Review 2
Stars: 5.0
Date: 2012-08-17 11:21:31
Text:
Just wow.  I had some pretty high expectations for Pane Rustica after reading all of the glowing Yelp reviews and also getting personal recommendations from others who have dined there.  Well, they not only met my expectations - they exceeded them.  Six of us visited for dinner a few weeks ago and we haven't stopped ta

In [17]:
from recommendation import recommend_restaurants

recommendations = recommend_restaurants(
    restaurants,
    reviews_by_business,
    cuisine="Italian",
    min_stars=4.0,
    min_reviews=50,
    keywords=["quiet", "romantic", "atmosphere", "service"],
    limit=5,
    reviews_per_restaurant=2
)

print(f"Recommendations returned: {len(recommendations)}")

for i, restaurant in enumerate(recommendations, start=1):
    print("\n" + "=" * 70)
    print(f"{i}. {restaurant['name']}")
    print(f"Rating: {restaurant['stars']}")
    print(f"Published reviews: {restaurant['review_count']}")

    for review in restaurant["reviews"]:
        print(f"\nReview ID: {review['review_id']}")
        print(f"Stars: {review['stars']}")
        print(f"Text: {review['text']}")

Recommendations returned: 5

1. Mirro's Pizzeria
Rating: 5.0
Published reviews: 111

Review ID: BI021ghwwy7XrsCv3Leg7w
Stars: 5.0
Text: Wow, Mirro's Pizzeria was simply extremely good. Not only was I impressed with the ambience and the atmosphere, but the food and service were great as well! Outstanding food, White Pie was magnificent undoubtedly. This White Pie was prepared just perfectly. Service is awesome, no complaints at all. Mirro's Pizzeria - They were terrific! I will certainly be back.

Review ID: mnaZ7Ri_T4GhwBoJJNXMTw
Stars: 5.0
Text: I am impressed, Mirro's Pizzeria was truly magnificent. Mirro's Pizzeria had great atmosphere, ambiance, cleanliness and customer service. Every single thing in the menu looked incredible, but I had to go with White Pie, it was an excellent meal. Staff ensured every thing was all right. Mirro's Pizzeria, so glad they're in town.

2. OLIVIA
Rating: 4.5
Published reviews: 545

Review ID: jJdSqbykDLi7Rz5F8OGgZQ
Stars: 5.0
Text: Lovely romantic at

In [20]:
from llm_client import ask_llm

response = ask_llm(
    "A restaurant is romantic but can get loud. "
    "Is it suitable for a quiet romantic dinner? "
    "Answer in two sentences."
)

print(response)

While this restaurant has a romantic ambiance, its tendency to get loud makes it unsuitable for a quiet dinner experience. For a truly peaceful romantic meal, a quieter venue would be preferable.


## PRINT PDF

In [ ]:
import os
from nbconvert import PDFExporter
import nbformat

NOTEBOOK_NAME = "tester_Ben_Prishtina.ipynb"
NOTEBOOK_PATH = NOTEBOOK_NAME

# Read the notebook as UTF-8
with open(NOTEBOOK_PATH, encoding="utf-8") as f:
    nb_node = nbformat.read(f, as_version=4)

# Set PDF document title
nb_node.metadata["title"] = "Appendix: CAP 6938 - Final Project Tester"

pdf_exporter = PDFExporter()
pdf_data, resources = pdf_exporter.from_notebook_node(nb_node)

NOTEBOOK_PDF = NOTEBOOK_PATH.rsplit('.', 1)[0] + '.pdf'
with open(NOTEBOOK_PDF, "wb") as f:
    f.write(pdf_data)
    
print(f"PDF CREATED: {NOTEBOOK_PDF}")